# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Lane 4 — CTR / Engagement Opportunity Scoring**

I'm going with this one mostly because I've basically built a version of it before. Last
semester I put together a momentum scanner over S&P 500 stocks — ran an OLS of return on beta,
then ranked stocks by the residual (actual return minus what that beta predicted). Lane 4 is the
same shape, just swap stocks for pages and beta for position tier: work out the CTR you'd expect
for a page given its tier (page-one pages should obviously click more than page-nine ones, same
way high-beta stocks should move more than low-beta ones), then rank pages by how far below that
expectation they land. The gap is the residual, same idea as before.

That's really why I picked it — not because I know search/SEO that well (I don't, honestly), but
because I already know the method is something I can get working in 7 weeks. Section 3 is me
checking the other half of the bet: is there actually a within-tier spread worth ranking, or does
CTR basically just track position, in which case this lane would just be re-deriving position and
nothing else.


In [1]:
import pandas as pd

# rate columns (ctr, engagement_rate, scroll_rate, ai_traffic_pct, trend_pct) are x100 percentages
# per the data dictionary -- e.g. ctr = 0.76 means 0.76%, not 76%.
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(df.shape)


(30000, 44)


## 2. The question: decision, action, cost of a wrong call

**Question:** among pages that already have real search visibility, which ones are
under-capturing clicks relative to what a page at their position tier normally earns, and should
get bumped to the front of the queue for a title, meta description, or snippet rewrite?

This is meant to feed one specific decision: which page a content reviewer opens first out of a
much bigger backlog of "visible but underperforming" pages. Their time is limited each week (say
the top 20-50 candidates on the list), so this is really a ranking problem, not a plain
good/bad classifier.

Who acts on it: an SEO/content reviewer with fixed weekly capacity. Once a page gets flagged,
they'll do one of: rewrite the title/meta, restructure the snippet, fix an intent mismatch, or —
if it turns out to be noise once they actually look — just monitor and move on.

Getting this wrong isn't free either way. A false positive (flagged as a gap, but it isn't really
one) means the reviewer burns real time on a page that was already fine, time that didn't go to a
page that needed it. A false negative (a real gap that never gets flagged) means a page with
strong impressions keeps under-converting quarter after quarter and nobody ever looks at it.
Since reviewer time is the actual scarce resource here, I think precision@K (of the top K pages
the queue surfaces, how many a reviewer would agree are real gaps) is the metric that matters,
not raw accuracy over all 30,000 rows.

Why not just write a rule? Something like "CTR < 0.5% is bad" ignores position completely — a
0.4% CTR is unremarkable at position 15 but is a real gap at position 2. So at minimum the rule
has to condition on position tier. Once it does, the actually interesting question is how much of
the leftover spread is explainable by other signals you'd have before the decision point (content
type, intent, freshness, word count, age). That's a tangled, multi-signal pattern that's easier to
learn than to hand-write, which is why I think this earns a model instead of staying an
if-statement.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [2]:
# Only pages with real position data count as "visible" here.
# avg_position == 0 means "no data" in this dataset, not rank zero -- so it must be excluded,
# not treated as a great position.
visible = df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 500)]
print(f"Visible pool (avg_position>0, impressions_90d>=500): {len(visible):,} of {len(df):,} rows")

# 1) Within-tier CTR spread: is CTR just a function of position tier, or is there real spread
#    left over once you condition on tier? (mean vs std, same units -- ctr is a x100 percentage)
tier_ctr = visible.groupby('position_tier')['ctr'].agg(['count', 'mean', 'std']).round(3)
print()
print("CTR by position tier (mean vs std -- spread on the same order as the mean means position")
print("alone does not explain most of the variation):")
print(tier_ctr)

# 2) How big is the "gap" pool inside the single largest, most reviewable tier (page_1)?
page1 = visible[visible['position_tier'] == 'page_1']
tier_mean = page1['ctr'].mean()
underperformers = page1[page1['ctr'] < tier_mean * 0.5]
print()
print(f"page_1 tier: {len(page1):,} visible pages, tier mean CTR = {tier_mean:.3f}%")
print(f"Pages sitting below half the tier's own mean CTR: {len(underperformers):,} "
      f"({100*len(underperformers)/len(page1):.1f}% of the tier)")
print(f"Median impressions_90d among those underperformers: "
      f"{underperformers['impressions_90d'].median():,.0f}")


Visible pool (avg_position>0, impressions_90d>=500): 16,726 of 30,000 rows

CTR by position tier (mean vs std -- spread on the same order as the mean means position
alone does not explain most of the variation):
               count   mean    std
position_tier                     
deep             389  0.043  0.140
page_1          7064  0.339  0.351
page_3_5        4330  0.143  0.185
striking        4485  0.267  0.316
top_3            458  0.347  0.422

page_1 tier: 7,064 visible pages, tier mean CTR = 0.339%
Pages sitting below half the tier's own mean CTR: 2,493 (35.3% of the tier)
Median impressions_90d among those underperformers: 3,409


## 4. Careful words: what I can and can't claim

What this can say by the end of 7 weeks: observed, position-adjusted associations — pages with
attributes X, Y, Z sit further below their tier's expected CTR than pages without them. It can
also produce a decision-support ranking — these N pages are the best candidates to review first
given limited reviewer time — backed by precision@K on a held-out set of clients. Language stays
directional the whole way through: "this suggests," "we observed," "associated with." Never
"causes."

What it can't say, no matter how good the model ends up looking: that rewriting a title or meta
description will actually raise a page's CTR. That's a causal claim, and there's no experiment in
this data (no A/B test, no before/after on a real edit) to back it up — the output is a candidate
list for a human to review, not a guaranteed fix. It also can't say anything about how Google's
ranking algorithm actually works — CTR and position are just observed outcomes here, not a window
into the algorithm. And it's silent on AI citations or AI search visibility entirely, that's out
of scope for this lane. Last thing: a low CTR isn't automatically a title/meta problem. Section 7
of the lane guide lists the look-alikes (consolidation, seasonality, SERP feature changes) I still
need to rule out before trusting a gap is real.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.